# Documents

A `Document` object in `LangChain` contains information about some data. A Document object has the following two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. You can use the metadata to track various details, such as the document ID, the file name, and other details.

In [ ]:
from langchain_core.documents import Document # Document is a container for text content with associated metadata
from langchain_community.document_loaders import PyPDFLoader # This loader is specifically designed to load and parse PDF files
from langchain_community.document_loaders import WebBaseLoader # This loader is designed to scrape and extract text content from web pages
from langchain_community.document_loaders import TextLoader # This loader is designed to load text from a text file
from langchain.text_splitter import CharacterTextSplitter # split document based on characters. Its simplest method for splitting text.
from langchain.text_splitter import RecursiveCharacterTextSplitter

import wget


In [ ]:
# Create a Document instance with content and metadata
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
metadata={
    'my_document_id' : 234234,                      # Unique identifier for this document
    'my_document_source' : "About Python",          # Source or title information
    'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
 }) # metadata is optional to create a document

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language.\n Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

## Document loaders

`Document loaders` load documents from a variety of sources; for instance, loading a PDF file and having the LLM read the PDF file using LangChain.

LangChain offers over 100 distinct document loaders, along with integrations with other major providers, such as AirByte and Unstructured. These integrations enable loading of all kinds of documents (HTML, PDF, code) from various locations including private Amazon S3 buckets, as well as from public websites).

You can find a list of document types that LangChain can load at [LangChain Document loaders](https://python.langchain.com/v0.1/docs/integrations/document_loaders/).

In this lab, you will use the PDF loader and the URL and website loader.


## PDF loader

load a PDF file as a Document object.

In [ ]:
# The loader will download the PDF from the specified URL and prepare it for loading
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

# Call the load() method to:
# 1. Download the PDF if needed
# 2. Extract text from each page
# 3. Create a list of Document objects, one for each page of the PDF
document = loader.load() # Download the PDF and Load as a langchain document

In [ ]:
document[2]  # take a look at the page 2

In [ ]:
print(document[1].page_content[:1000])  # print the page 1's first 1000 tokens

## URL and website loader

load content from a URL or website into a Document object:

In [ ]:
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/") # Create an instance by passing the URL of the web page to load

# Call the load() method to:
# 1. Send an HTTP request to the specified URL
# 2. Download the HTML content
# 3. Parse the HTML to extract meaningful text
# 4. Create a list of Document objects containing the extracted content
web_data = loader.load()

first_document = web_data[0]
print(first_document.page_content[:1000]) # Print the first 1000 characters of the page content from the first Document

## Text Loader

In [ ]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')


file downloaded


In [ ]:
loader = TextLoader(filename)
documents = loader.load()

print(documents[0].page_content[:1000]) 

 regulations. Employees may be subject to testing in cases of reasonable suspicion, post-accident, or as part of routine workplace safety measures.
Reporting: Employees should report any concerns related to drug or alcohol misuse by themselves or their colleagues, as well as safety concerns arising from such misuse.
Treatment and Assistance: Employees with substance abuse issues are encouraged to seek help. The organization is committed to providing support, resources, and information to assist those seeking treatment.
Consequences: Violation of this policy may result in disciplinary actions, up to and including termination of employment. Legal action may also be pursued when necessary.
Policy Review: This policy will undergo periodic review to ensure its continued relevance and compliance with evolving legal requirements and best practices for a safe and productive work environment.
Your adherence to this policy is appreciated as it helps to maintain a safe and drug-free workplace for

## Text splitters
 
Split a long document into smaller and manageable chunks.

In [ ]:
# Create a CharacterTextSplitter with specific configuration:
# - chunk_size=200: Each chunk will contain approximately 200 characters
# - chunk_overlap=20: Consecutive chunks will overlap by 20 characters to maintain context
# - separator="\n": Text will be split at newline characters when possible - By default its value is "\n\n"
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

# Split the previously loaded document (PDF or other text) into chunks
# - It returns a new list of Document objects where each contains a chunk of text
# - It preserves the original metadata for each chunk
chunks = text_splitter.split_documents(document)

print(len(chunks)) # Length of chunks

In [ ]:
chunks[5].page_content   # take a look at any chunk's page content

### Exercise - Working with Document Loaders and Text Splitters

In [ ]:
# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = PyPDFLoader(web_url)
web_document = loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document) # Chunks of PDF document
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs) # Total characters in all chunks
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 6th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs] # list of total characters in all chunks
        min_len = min(lengths) # minimum characters in a chunk
        max_len = max(lengths) # maximun characters in a chunk
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")